# AutoData lab

> **SealQA question:** What is the current age of the oldest person to sail solo across the Pacific Ocean?


**Terminology.** The evolving `meta.Text` is a data-generation *recipe*. A `restricted_solver` sees only the generated question; an `informed_solver` sees the same question plus selected frozen evidence. The paper describes weak/strong models; this lab uses one model under different contexts. Inspired by [AutoData](https://facebookresearch.github.io/RAM/blogs/autodata/).

## What changes

Only the recipe changes. Each recipe uses a generator and two solver sessions, followed by local schema, answerability, leakage, and normalized exact-answer checks. One rewrite separates the two evaluations: **3 + 1 + 3 = 7 sessions**. Same-model/different-context is a teaching simplification and a confounder.

## Live declaration

Bring your own user-owned SDK wrapper and identify its behavior-affecting declaration. Set `META_EVOLVE_LIVE_SDK=1` only after authenticating it and reviewing its session, tool, permission, and spending limits. Every inner run declares `INNER_BUDGET.wall_seconds = 1860`; the wrapper must honor that bound or refuse preflight. These calls may incur cost.

In [ ]:
from pathlib import Path
import os, sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
lab = root / "examples/research/sealqa_labs"
if str(lab) not in sys.path:
    sys.path.insert(0, str(lab))

from sealqa_labs import UserSdkSessionRunner
from sealqa_labs.config import *

print({
    "wrapper": WRAPPER_DECLARATION, "sessions": TOTAL_SESSIONS,
    "inner_wall_seconds": INNER_BUDGET.wall_seconds,
    "opt_in": f"{LIVE_OPT_IN}=1",
})


In [ ]:
if os.getenv(LIVE_OPT_IN) != "1":
    raise RuntimeError(
        f"Live-only lab: set {LIVE_OPT_IN}=1 after authenticating your SDK."
    )

from sealqa_labs.autodata import execute

raise RuntimeError(
    "Define build_agent(request), then run(UserSdkSessionRunner(agent_factory=build_agent))."
)
print("sessions executed:", len(result.sessions))
print("usage:", result.usage)


## Visible record

Generated items, both solver answers, acceptance decisions, rejection reasons, recipe lineage, typed failures, and usage remain inspectable. A session failure is unrankable; it is never converted to a rejected example with score zero.

In [ ]:
for session in result.sessions:
    print(session.label, "failure=" + (session.failure.kind if session.failure else "none"))
print("artifacts:", result.artifacts)
print("measurements:", result.measurements)


## Live observation

No official provider capture is pending. A user may perform one authorized run with a declared wrapper. Preserve rejection or a successor that is no better; do not rerun until a generated item happens to pass.

## Audit and exercise

- Does the restricted solver workspace contain evidence?
- Can the informed solver's context advantage be confused with model strength?
- Does the generated reference answer leak verbatim into its question?

**Change and predict:** require two evidence document IDs. Predict which deterministic check changes and whether acceptance becomes easier or harder.